In [2]:
"""
EDA for the skin condition classifier — class counts, corrupted file check,
dimension check, and a basic color comparison for melanoma vs nevus.

Run this from your project root with the venv active:
    python eda.py

IMPORTANT: the folder paths below are guesses based on what we've discussed.
Check each path actually exists on your machine before trusting the output —
adjust anything under "# EDIT THIS" if your folder names differ.

SCOPE (locked):
    Target classes: melanoma, nevus, bcc, seborrheic_keratosis,
                    acne, eczema, vitiligo, nail_psoriasis
    Tinea:          EXCLUDED (combined total 316 — too small vs next smallest 1542)
    SkinDisNet:     only Eczema included; all other diagnoses out of scope
"""

import os
import pandas as pd
import numpy as np
from PIL import Image
import glob
import hashlib
import json
import random

BASE = r"C:\Users\TOSHIBA\Desktop\skin_Conditions_AI\data\raw"

# ============================================================
# PART 1: CLASS COUNTS
# ============================================================
# Different datasets store their labels differently, so this dict tells
# the script HOW to count each one. "folder" means: count files inside
# that exact folder. We build this list manually because there's no one
# single way to ask "how many of class X do I have" across 4 datasets.

print("=" * 60)
print("PART 1: Class counts")
print("=" * 60)

class_counts = {}

# --- ISIC (clean folder-per-class) ---
isic_path = os.path.join(BASE, "isic_archive")
for class_name in os.listdir(isic_path):
    full_path = os.path.join(isic_path, class_name)
    if os.path.isdir(full_path):
        class_counts[class_name] = len(os.listdir(full_path))

# --- HAM10000 (CSV-based, not folders) ---
# The 'dx' column holds short codes. We map them to readable names and
# ADD to whatever ISIC already gave us, since both datasets contribute
# to some of the same classes (e.g. melanoma appears in both).
ham_csv = os.path.join(BASE, "ham10000", "HAM10000_metadata.csv")
ham_df = pd.read_csv(ham_csv)
dx_map = {
    "mel": "melanoma",
    "nv": "nevus",
    "bcc": "bcc",
    "bkl": "seborrheic_keratosis",  # benign keratosis-like lesions
    # akiec, df, vasc aren't on our class list — skipped on purpose
}
ham_counts = ham_df["dx"].value_counts()
for code, mapped_name in dx_map.items():
    if code in ham_counts:
        class_counts[mapped_name] = class_counts.get(mapped_name, 0) + int(ham_counts[code])

# --- ACNE04 (sum severity grade folders into one 'acne' class) ---
acne_path = os.path.join(BASE, "acne04")
acne_total = 0
for grade_folder in ["acne0_1024", "acne1_1024", "acne2_1024", "acne3_1024"]:
    grade_path = os.path.join(acne_path, grade_folder)
    if os.path.isdir(grade_path):
        acne_total += len(os.listdir(grade_path))
class_counts["acne"] = acne_total

# --- Extra acne from Skin Disease Classification Dataset ---
extra_acne_path = os.path.join(BASE, "Skin Disease Classification Dataset", "acne")
if os.path.isdir(extra_acne_path):
    class_counts["acne"] += len(os.listdir(extra_acne_path))

# --- DermNet (ONLY the clean eczema folder — everything else excluded) ---
dermnet_path = os.path.join(BASE, "dermnet")
eczema_total = 0
for split in ["train", "test"]:
    split_path = os.path.join(dermnet_path, split, "Eczema Photos")
    if os.path.isdir(split_path):
        eczema_total += len(os.listdir(split_path))
class_counts["eczema"] = eczema_total

# --- Vitiligo (combining two sources) ---
vitiligo_total = 0

skindiseasebd_vitiligo = os.path.join(
    BASE, "SkinDiseaseBD A Dataset of Common Skin Disease Ima",
    "Updated Images", "Vitiligo"
)
if os.path.isdir(skindiseasebd_vitiligo):
    vitiligo_total += len(os.listdir(skindiseasebd_vitiligo))

classification_vitiligo = os.path.join(
    BASE, "Skin Disease Classification Dataset", "Vitiligo"
)
if os.path.isdir(classification_vitiligo):
    vitiligo_total += len(os.listdir(classification_vitiligo))

class_counts["vitiligo"] = vitiligo_total

# --- TINEA: EXCLUDED ---
# Tinea was originally sourced from SkinDiseaseBD's "Tinea Ringworm" folder
# and SkinDisNet's "Tinea Corporis" subset. Combined total came to 316 images,
# ~5x smaller than the next smallest class (acne: 1542). Removed from the
# target class list. Documented scope reduction, not an oversight.
#
# The SkinDiseaseBD tinea folder and SkinDisNet Tinea Corporis rows are
# therefore NOT counted here — see SkinDisNet block below for skip logging.

# --- Nail psoriasis (renamed from general "psoriasis" — honest scope) ---
nail_psoriasis_path = os.path.join(
    BASE, "Skin Disease Classification Dataset", "Nail_psoriasis"
)
if os.path.isdir(nail_psoriasis_path):
    class_counts["nail_psoriasis"] = len(os.listdir(nail_psoriasis_path))

# ============================================================
# SkinDisNet (Mendeley) — SCOPE DECISION (documented)
# ============================================================
# SkinDisNet contains 6 diagnoses: Atopic Dermatitis, Contact Dermatitis,
# Eczema, Scabies, Seborrheic Dermatitis, Tinea Corporis.
#
# ONLY Eczema is pulled in — the only diagnosis that overlaps with our
# target class list. The other five are excluded as OUT OF SCOPE.
#
# NOTE on naming: SkinDisNet's "Seborrheic Dermatitis" is NOT the same
# condition as our existing "seborrheic_keratosis" (HAM10000 'bkl').
# They are different diseases and must not be merged.

sdn_dirs = glob.glob(os.path.join(BASE, "SkinDisNet*"))
sdn_root = sdn_dirs[0] if sdn_dirs else None
sdn_df = None

if sdn_root is None:
    print("WARNING: no SkinDisNet folder found under", BASE)
else:
    sdn_csvs = glob.glob(
        os.path.join(sdn_root, "**", "SkinDisNet_Metadata.csv"), recursive=True
    )
    if not sdn_csvs:
        print("WARNING: SkinDisNet_Metadata.csv not found inside", sdn_root)
    else:
        sdn_df = pd.read_csv(sdn_csvs[0])
        sdn_df["Diagnosis"] = sdn_df["Diagnosis"].str.strip()

        sdn_map = {"Eczema": "eczema"}
        sdn_counts = sdn_df["Diagnosis"].value_counts()

        added_per_class = {}
        for sdn_name, mapped_name in sdn_map.items():
            n = int(sdn_counts.get(sdn_name, 0))
            class_counts[mapped_name] = class_counts.get(mapped_name, 0) + n
            added_per_class[mapped_name] = n

        skipped = {k: int(v) for k, v in sdn_counts.items() if k not in sdn_map}

        total_added = sum(added_per_class.values())
        total_skipped = sum(skipped.values())

        print(f"\nSkinDisNet: added {total_added} images "
              f"({', '.join(f'{k}={v}' for k, v in added_per_class.items())})")
        print(f"SkinDisNet: skipped {total_skipped} images (out of scope):")
        for diag, n in skipped.items():
            print(f"    - {diag}: {n}")

        sdn_summary = {
            "source": "SkinDisNet (Mendeley)",
            "scope_decision": ("only Eczema included; Tinea Corporis, Atopic "
                               "Dermatitis, Contact Dermatitis, Scabies, and "
                               "Seborrheic Dermatitis excluded"),
            "added": added_per_class,
            "skipped": skipped,
            "note_on_naming": ("SkinDisNet 'Seborrheic Dermatitis' is a distinct "
                               "condition from HAM10000 'seborrheic_keratosis' "
                               "and was NOT merged."),
            "note_on_tinea": ("Tinea Corporis is listed as skipped here, but it "
                              "was also dropped from the target class list entirely "
                              "due to insufficient total images (316 combined)."),
        }
        out_path = os.path.join(BASE, "..", "skindisnet_scope_summary.json")
        with open(out_path, "w") as f:
            json.dump(sdn_summary, f, indent=2)
        print(f"SkinDisNet scope summary written to: {os.path.abspath(out_path)}")

# --- Final class counts ---
print("\nCurrent class counts (final target set):")
for cls, count in sorted(class_counts.items()):
    print(f"  {cls}: {count}")

print(f"\nTotal images across all classes: {sum(class_counts.values())}")
print(f"Number of classes: {len(class_counts)}")

# ============================================================
# PART 1b: NEVUS CAP
# ============================================================
# nevus is ~27107 vs ~2000 for every other class. We subsample to 2500
# to avoid a degenerate classifier. Files are NOT deleted — we save a
# reproducible list of kept filenames so training can filter by it.

print("=" * 60)
print("PART 1b: Nevus subsampling")
print("=" * 60)

NEVUS_CAP = 2500
NEVUS_DIR = os.path.join(isic_path, "nevus")

if os.path.isdir(NEVUS_DIR):
    all_nevus = [f for f in os.listdir(NEVUS_DIR)
                 if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    print(f"Total nevus files on disk: {len(all_nevus)}")

    if len(all_nevus) > NEVUS_CAP:
        random.seed(42)
        keep = random.sample(all_nevus, NEVUS_CAP)
        nevus_out = os.path.join(BASE, "..", "nevus_cap_2500.json")
        with open(nevus_out, "w") as f:
            json.dump(sorted(keep), f)
        print(f"Subsampled to {NEVUS_CAP}. Kept list saved to: "
              f"{os.path.abspath(nevus_out)}")
        print("NOTE: files were NOT deleted — training should filter by this list.")
    else:
        print(f"Fewer than {NEVUS_CAP} nevus files found, no capping needed.")
else:
    print(f"WARNING: nevus folder not found at {NEVUS_DIR}")

# ============================================================
# PART 2: CORRUPTED FILE SCAN
# ============================================================
# We open every image and see if it fails. Better to find this now than
# mid-training. This can take a few minutes on large folders — that's normal.

print("=" * 60)
print("PART 2: Scanning for corrupted files")
print("=" * 60)

corrupted = []
scanned = 0

for class_name in os.listdir(isic_path):
    folder = os.path.join(isic_path, class_name)
    if not os.path.isdir(folder):
        continue
    for fname in os.listdir(folder):
        fpath = os.path.join(folder, fname)
        try:
            img = Image.open(fpath)
            img.verify()
            scanned += 1
        except Exception:
            corrupted.append(fpath)

print(f"Scanned {scanned} ISIC images. Found {len(corrupted)} corrupted.")
if corrupted:
    print("Corrupted files:", corrupted[:10], "..." if len(corrupted) > 10 else "")

# ============================================================
# PART 3: IMAGE DIMENSIONS
# ============================================================

print("=" * 60)
print("PART 3: Image dimensions (ISIC sample)")
print("=" * 60)

sizes = []
sample_count = 0
for class_name in os.listdir(isic_path):
    folder = os.path.join(isic_path, class_name)
    if not os.path.isdir(folder):
        continue
    for fname in os.listdir(folder)[:50]:
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        fpath = os.path.join(folder, fname)
        try:
            with Image.open(fpath) as img:
                sizes.append(img.size)
            sample_count += 1
        except Exception:
            pass

if sizes:
    widths = [s[0] for s in sizes]
    heights = [s[1] for s in sizes]
    print(f"Sampled {sample_count} images.")
    print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
    print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
else:
    print("No images sampled — check ISIC path.")

# ============================================================
# PART 4: BASIC COLOR ANALYSIS — melanoma vs nevus
# ============================================================

print("=" * 60)
print("PART 4: Color analysis — melanoma vs nevus")
print("=" * 60)

def channel_stats(folder_path, sample_size=30):
    files = [f for f in os.listdir(folder_path)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))][:sample_size]
    r_vals, g_vals, b_vals = [], [], []
    for fname in files:
        fpath = os.path.join(folder_path, fname)
        try:
            img = Image.open(fpath).convert("RGB")
            arr = np.array(img)
            r_vals.append(arr[:, :, 0].std())
            g_vals.append(arr[:, :, 1].std())
            b_vals.append(arr[:, :, 2].std())
        except Exception:
            pass
    if not r_vals:
        return (0.0, 0.0, 0.0)
    return np.mean(r_vals), np.mean(g_vals), np.mean(b_vals)

def find_ham_image_path(image_id):
    for part in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        candidate = os.path.join(BASE, "ham10000", part, image_id + ".jpg")
        if os.path.exists(candidate):
            return candidate
    return None

def ham_channel_stats(dx_code, sample_size=30):
    subset = ham_df[ham_df["dx"] == dx_code].head(sample_size)
    r_vals, g_vals, b_vals = [], [], []
    for image_id in subset["image_id"]:
        fpath = find_ham_image_path(image_id)
        if fpath is None:
            continue
        try:
            img = Image.open(fpath).convert("RGB")
            arr = np.array(img)
            r_vals.append(arr[:, :, 0].std())
            g_vals.append(arr[:, :, 1].std())
            b_vals.append(arr[:, :, 2].std())
        except Exception:
            pass
    if not r_vals:
        return (0.0, 0.0, 0.0)
    return np.mean(r_vals), np.mean(g_vals), np.mean(b_vals)

melanoma_path = os.path.join(isic_path, "melanoma")
nevus_path = os.path.join(isic_path, "nevus")

ham_mel_r, ham_mel_g, ham_mel_b = ham_channel_stats("mel")
ham_nv_r, ham_nv_g, ham_nv_b = ham_channel_stats("nv")
mel_r, mel_g, mel_b = channel_stats(melanoma_path)
nv_r, nv_g, nv_b = channel_stats(nevus_path)

print(f"HAM10000 Melanoma — avg channel std (R,G,B): "
      f"({ham_mel_r:.1f}, {ham_mel_g:.1f}, {ham_mel_b:.1f})")
print(f"HAM10000 Nevus    — avg channel std (R,G,B): "
      f"({ham_nv_r:.1f}, {ham_nv_g:.1f}, {ham_nv_b:.1f})")
print(f"ISIC Melanoma — avg channel std (R,G,B): "
      f"({mel_r:.1f}, {mel_g:.1f}, {mel_b:.1f})")
print(f"ISIC Nevus    — avg channel std (R,G,B): "
      f"({nv_r:.1f}, {nv_g:.1f}, {nv_b:.1f})")
print("\nHigher numbers = more color variation within images of that class.")
print("If melanoma's numbers are noticeably higher, that matches the expected")
print("'color variegation' pattern dermatologists use — a good sign your data")
print("carries a real, learnable signal for this.")

# ============================================================
# PART 5: Duplicate check — ISIC vs HAM10000
# ============================================================

print("=" * 60)
print("PART 5: Checking for duplicate images (ISIC vs HAM10000)")
print("=" * 60)

def file_hash(filepath):
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

isic_hashes = {}
for class_name in ["melanoma", "nevus", "bcc", "seborrheic_keratosis"]:
    folder = os.path.join(isic_path, class_name)
    if not os.path.isdir(folder):
        continue
    for fname in os.listdir(folder):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            fpath = os.path.join(folder, fname)
            isic_hashes[file_hash(fpath)] = fpath

print("ISIC hashes collected:", len(isic_hashes))

duplicates = []
ham_files_checked = 0
for part in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
    part_path = os.path.join(BASE, "ham10000", part)
    if not os.path.isdir(part_path):
        continue
    for fname in os.listdir(part_path):
        fpath = os.path.join(part_path, fname)
        ham_files_checked += 1
        h = file_hash(fpath)
        if h in isic_hashes:
            duplicates.append({
                "ham10000_file": fpath,
                "matches_isic_file": isic_hashes[h],
            })

print("HAM10000 files checked:", ham_files_checked)
print(f"Found {len(duplicates)} duplicates")

dup_out = os.path.join(BASE, "..", "duplicate_exclusions.json")
with open(dup_out, "w") as f:
    json.dump(duplicates, f, indent=2)
print(f"Duplicate list saved to: {os.path.abspath(dup_out)}")

# ============================================================
# PART 6: DDI metadata peek
# ============================================================

print("=" * 60)
print("PART 6: DDI metadata")
print("=" * 60)

ddi_csv = os.path.join(BASE, "DDI", "images", "ddi_metadata.csv")
if os.path.exists(ddi_csv):
    ddi_df = pd.read_csv(ddi_csv)
    print("\nSkin tone distribution:")
    print(ddi_df["skin_tone"].value_counts())
    print("\nDisease distribution:")
    print(ddi_df["disease"].value_counts())
else:
    print(f"DDI metadata not found at {ddi_csv}")

print("\n" + "=" * 60)
print("EDA COMPLETE")
print("=" * 60)

PART 1: Class counts

SkinDisNet: added 466 images (eczema=466)
SkinDisNet: skipped 1244 images (out of scope):
    - Contact Dermatitis: 477
    - Scabies: 343
    - Tinea Corporis: 275
    - Seborrheic Dermatitis: 79
    - Atopic Dermatitis: 70
SkinDisNet scope summary written to: C:\Users\TOSHIBA\Desktop\skin_Conditions_AI\data\skindisnet_scope_summary.json

Current class counts (final target set):
  acne: 1542
  bcc: 2017
  eczema: 2010
  melanoma: 2616
  nail_psoriasis: 2520
  nevus: 27107
  seborrheic_keratosis: 2602
  vitiligo: 2328

Total images across all classes: 42742
Number of classes: 8
PART 1b: Nevus subsampling
Total nevus files on disk: 20399
Subsampled to 2500. Kept list saved to: C:\Users\TOSHIBA\Desktop\skin_Conditions_AI\data\nevus_cap_2500.json
NOTE: files were NOT deleted — training should filter by this list.
PART 2: Scanning for corrupted files
Scanned 24899 ISIC images. Found 12 corrupted.
Corrupted files: ['C:\\Users\\TOSHIBA\\Desktop\\skin_Conditions_AI\\data